In [31]:
!pip install -U pip transformers

In [32]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [33]:
checkpoint = 'facebook/nllb-200-distilled-600M'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [34]:
print(f"{len(tokenizer.vocab)}\n")

tokenizer.vocab

256204



{'▁సహజ': 184427,
 '▁mhi': 63348,
 '▁پا': 13035,
 '▁tirhisiwa': 111186,
 'nyddol': 221124,
 '機関の': 184505,
 '▁جریان': 80232,
 'ક્ક': 57940,
 'ない': 968,
 'ukwa': 6851,
 '▁бирү': 164377,
 'દ્ર': 63717,
 '▁усь': 160276,
 '▁zou': 14058,
 '▁seksu': 32801,
 '▁rer': 59037,
 '▁reeds': 72425,
 '▁šķēr': 183409,
 '判': 250309,
 '▁제공하고': 157445,
 '▁partena': 241965,
 '▁casos': 39619,
 '▁vato': 133087,
 '▁draußen': 212500,
 'መናዊ': 245358,
 '▁पत्रिका': 173348,
 'apun': 15418,
 '▁finanzjament': 178257,
 '▁rade': 72666,
 '▁йй': 193760,
 '▁kounbé': 174661,
 '▁serê': 222266,
 '▁آتاسي': 187136,
 'җиза': 188392,
 '▁Mariam': 230661,
 '▁Mga': 24585,
 '渴': 254615,
 'Israyeli': 39600,
 '▁ป': 32189,
 '▁yılında': 80384,
 'pech': 114183,
 '▁ماڻهن': 75065,
 'மாய்': 171379,
 '▁tumut': 119119,
 '▁چاہ': 15010,
 'lagip': 215105,
 '▁nege': 114228,
 '▁nødt': 97890,
 'vità': 184506,
 'ള': 248849,
 '▁nettet': 168215,
 '▁būt': 23530,
 '▁ວ່າ': 20806,
 '▁njupuk': 118592,
 '▁ұсы': 39304,
 'ambira': 29438,
 'くない': 51311,
 '▁rat

In [35]:
thai_char_min = 0x0E00
thai_char_max = 0x0E7F

thai_tokens = [
    token for token in tokenizer.vocab.keys()
    if any(thai_char_min <= ord(char) <= thai_char_max for char in token)
]

thai_token_count = len(thai_tokens)
sample_size = 20
thai_tokens_sample = thai_tokens[:sample_size]


print(f"{thai_token_count}\n")
for token in thai_tokens_sample:
  print(token)


1712

็ค
คนอื่น
เรียน
เร็ว
เค้า
กร
ถูก
ตอนนี้
ระเบ
ซิ
แข
ก็ตาม
▁ข้อ
เคร
▁สําหรับ
วิธี
็บ
คล
▁พวกเรา
บ้าง


In [36]:
import tensorflow as tf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import math

In [37]:
sentence = 'Work hard, play harder'

In [38]:
cleaned_sentence = sentence.replace(',', '')
cleaned_sentence

'Work hard play harder'

In [39]:
words = cleaned_sentence.split()
words

['Work', 'hard', 'play', 'harder']

In [40]:
sorted_words = sorted(words)
sorted_words

['Work', 'hard', 'harder', 'play']

In [41]:
dc = {word: index for index, word in enumerate(sorted_words)}
dc

{'Work': 0, 'hard': 1, 'harder': 2, 'play': 3}

In [42]:
sentence_int = tf.constant(
    [dc[s] for s in sentence.replace(',', '').split()],
    dtype=tf.int32
)

In [43]:
print(sentence)
print(sentence_int)

Work hard, play harder
tf.Tensor([0 1 3 2], shape=(4,), dtype=int32)


In [44]:
# สร้าง embedding layer
tf.random.set_seed(123)
vocab_size = 50_000
embedding_dim = 2

embed = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)

In [45]:
embedded_sentence = embed(sentence_int)

In [46]:
embedded_sentence

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.00870323,  0.04908073],
       [-0.03102988, -0.03036381],
       [-0.02166357,  0.03601504],
       [ 0.01006973, -0.04249501]], dtype=float32)>

In [47]:
tf.random.set_seed(123)
vocab_size = 50_000
embedding_dim = 2

dummy_input = tf.constant([0, 1, 2], dtype=tf.int32)

# Case 1 Default initializer (RandomUniform(-0.05, 0.05))
embed_default = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)
_ = embed_default(dummy_input) # เรียกใช้งาน layer เพื่อสร้าง weights
weights_default = embed_default.get_weights()[0].flatten()
weights_default.shape

(100000,)

In [48]:
# Case 2 GlorotUniform initializer
tf.random.set_seed(123)
embed_glorot = tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    embeddings_initializer=tf.keras.initializers.GlorotUniform()
)
_ = embed_glorot(dummy_input) # เรียกใช้งาน layer เพื่อสร้าง weights
weights_glorot = embed_glorot.get_weights()[0].flatten()
weights_glorot.shape

(100000,)

In [49]:
fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Histogram(x=weights_default, nbinsx=50, name="Default Uniform [-0.05, 0.05]", opacity=0.6))
fig.add_trace(go.Histogram(x=weights_glorot, nbinsx=50, name="Glorot Uniform", opacity=0.6))

fig.update_layout(
    title_text='Embedding Layer Initialization Comparison',
    xaxis_title_text='Weight values',
    yaxis_title_text='Frequency',
    barmode='overlay',
    legend_orientation="h",
    legend_yanchor="bottom",
    legend_y=1.02,
    legend_xanchor="right",
    legend_x=1
)

fig.show()

print("Default initializer range ", weights_default.min(), weights_default.max())
print("Glorot initializer range ", weights_glorot.min(), weights_glorot.max())

Default initializer range  -0.049999177 0.049999762
Glorot initializer range  -0.010953979 0.010953934


In [50]:
def glorot_uniform_limits(fan_in, fan_out):
    limit = math.sqrt(6.0 / (fan_in + fan_out))
    a, b = -limit, limit
    return a, b

# ตัวอย่าง Embedding layer (vocab_size=50000, embedding_dim=2)
fan_in = 50000
fan_out = 2

a, b = glorot_uniform_limits(fan_in, fan_out)
print("Glorot Uniform a =", a)
print("Glorot Uniform b =", b)

Glorot Uniform a = -0.010954232067652772
Glorot Uniform b = 0.010954232067652772


In [51]:
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [52]:
token_embedding_layer = model.model.encoder.embed_tokens
token_embedding_layer.weight.shape

torch.Size([256206, 1024])

In [53]:
long_sentence = "In the vast realm of natural language processing, understanding the nuances of how models handle sequential data is crucial. Positional encoding plays a vital role in providing this essential information to the model, allowing it to differentiate between words at different positions in a sentence, which is fundamental for tasks like translation, summarization, and text generation."

In [54]:
tokens = tokenizer(long_sentence, return_tensors="pt")

print(tokens['input_ids'][0])

tensor([256047,    717,    349,  14430,  12284, 248070,    452,  25307,  65445,
        157278, 248079, 133930,    349,    713,  75831,    452,  11657, 141057,
         47274, 116914, 124785,   6067,    248, 182071, 248075,  12013,  58409,
         12025, 246156,   3054,    705,      9, 104781,  76065,    108, 174693,
          3423, 140515,  18781,    202,    349,  14916, 248079,  82935,     87,
           796,    202,  53054,    502,  25914,  51744,    230,  30158, 199073,
           108,      9, 109267, 248079,   9089,    248,  75529,    351, 226047,
          6399, 200356, 248079,   2493, 109207, 181953, 248079,    540,  35883,
        120531, 248075,      2])


In [55]:
len(tokens['input_ids'][0])

75

In [56]:
token_embedding_layer(tokens['input_ids'][0][0]).shape

torch.Size([1024])

In [57]:
token_embeddings = token_embedding_layer(tokens['input_ids'][0])

print("Token Embedding Matrix shape", token_embeddings.shape)
token_embeddings

Token Embedding Matrix shape torch.Size([75, 1024])


tensor([[-5.0000e+00, -1.2725e+00, -9.3604e-01,  ..., -1.8297e+01,
         -9.1328e+00, -1.0672e+01],
        [ 2.6416e-01,  2.6831e-01,  2.0117e-01,  ...,  3.2715e+00,
         -3.2402e+00,  3.1738e+00],
        [ 4.3579e-01, -2.3352e-01,  2.6825e-02,  ...,  5.4648e+00,
          2.7129e+00,  5.5430e+00],
        ...,
        [ 8.5859e+00, -4.5391e+00, -4.7314e-01,  ..., -7.9529e-02,
          7.4844e+00, -7.5156e+00],
        [-2.4863e+00, -2.7515e-01,  5.6114e-03,  ...,  1.0180e+01,
         -7.2422e+00, -4.8047e+00],
        [-7.8320e-01, -9.0527e-01, -9.4482e-01,  ...,  3.1078e+01,
         -8.1494e-01, -8.7354e-01]], grad_fn=<MulBackward0>)

In [58]:
import plotly.express as px

token_embeddings_np = token_embeddings.detach().numpy()

fig = px.imshow(
    token_embeddings_np,
    color_continuous_scale="RdBu",
    labels=dict(x="Embedding Dimension", y="Token Index", color="Value"),
    title="Token Embedding Heatmap"
)

fig.update_xaxes(side="top")
fig.update_layout(height=500, width=900)
fig.show()

In [59]:
d = embedded_sentence.shape[-1]
d

2

In [60]:
d_q, d_k, d_v = 2, 2, 4

d_q, d_k, d_v

(2, 2, 4)

In [61]:
tf.random.set_seed(123)
W_query = tf.Variable(tf.random.uniform((d, d_q)), trainable=True)
W_key   = tf.Variable(tf.random.uniform((d, d_k)), trainable=True)
W_value = tf.Variable(tf.random.uniform((d, d_v)), trainable=True)

print(W_query.shape, W_key.shape, W_value.shape)

(2, 2) (2, 2) (2, 4)


In [62]:
W_query

<tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[0.12615311, 0.5727513 ],
       [0.2993133 , 0.5461836 ]], dtype=float32)>

In [63]:
W_key

<tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[0.88968754, 0.12354946],
       [0.7718717 , 0.6850728 ]], dtype=float32)>

In [64]:
W_value

<tf.Variable 'Variable:0' shape=(2, 4) dtype=float32, numpy=
array([[0.48962688, 0.5857923 , 0.36451697, 0.6550509 ],
       [0.9075084 , 0.37557673, 0.6882372 , 0.25384045]], dtype=float32)>

In [65]:
embedded_sentence

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.00870323,  0.04908073],
       [-0.03102988, -0.03036381],
       [-0.02166357,  0.03601504],
       [ 0.01006973, -0.04249501]], dtype=float32)>

In [66]:
queries = tf.matmul(embedded_sentence, W_query)
keys    = tf.matmul(embedded_sentence, W_key)
values  = tf.matmul(embedded_sentence, W_value)

In [67]:
print("Queries shape", queries.shape)
queries

Queries shape (4, 2)


<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.01578846,  0.03179188],
       [-0.01300281, -0.03435662],
       [ 0.00804686,  0.00726299],
       [-0.011449  , -0.01744263]], dtype=float32)>

In [68]:
print("Keys shape", keys.shape)
keys

Keys shape (4, 2)


<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.04562718,  0.03469915],
       [-0.05104386, -0.02463515],
       [ 0.00852519,  0.0219964 ],
       [-0.02384179, -0.02786807]], dtype=float32)>

In [69]:
print("Values shape", values.shape)
values

Values shape (4, 4)


<tf.Tensor: shape=(4, 4), dtype=float32, numpy=
array([[ 0.04880251,  0.02353187,  0.03695166,  0.01815974],
       [-0.04274848, -0.029581  , -0.03220842, -0.02803371],
       [ 0.02207689,  0.00083606,  0.01689015, -0.00504867],
       [-0.03363417, -0.01006137, -0.02557606, -0.00419077]],
      dtype=float32)>

In [70]:
omega = tf.matmul(queries, keys, transpose_b=True)

print("Omega shape", omega.shape)
print("Omega (Unnormalized attention weights)")
print(omega)

Omega shape (4, 4)
Omega (Unnormalized attention weights)
tf.Tensor(
[[ 0.00182353 -0.0015891   0.00083391 -0.0012624 ]
 [-0.00178543  0.00151009 -0.00086657  0.00126746]
 [ 0.00061917 -0.00058967  0.00022836 -0.00039426]
 [-0.00112763  0.0010141  -0.00048128  0.00075906]], shape=(4, 4), dtype=float32)


In [71]:
d_k = tf.cast(d_k, tf.float32)

scaled_omega = omega / tf.sqrt(d_k)

attention_weights = tf.nn.softmax(scaled_omega, axis=-1)

print("Attention Weights")
print(attention_weights)

Attention Weights
tf.Tensor(
[[0.250331   0.24972768 0.2501559  0.24978538]
 [0.24967892 0.25026143 0.2498412  0.25021848]
 [0.2501155  0.2499018  0.2500464  0.24993634]
 [0.24979344 0.25017202 0.24990764 0.2501269 ]], shape=(4, 4), dtype=float32)


In [72]:
row_sums = tf.reduce_sum(attention_weights, axis=-1)

print("Sum of each row in attention_weights")
row_sums

Sum of each row in attention_weights


<tf.Tensor: shape=(4,), dtype=float32, numpy=array([1., 1., 1., 1.], dtype=float32)>

In [73]:
context_vector = tf.matmul(attention_weights, values)

print("Context Vector shape", context_vector.shape)
print(context_vector)

Context Vector shape (4, 4)
tf.Tensor(
[[-0.00133735 -0.00380048 -0.00095654 -0.0047646 ]
 [-0.00141351 -0.00383623 -0.00101422 -0.00479163]
 [-0.00136281 -0.00381231 -0.00097582 -0.00477347]
 [-0.00139955 -0.00382991 -0.00100365 -0.00478699]], shape=(4, 4), dtype=float32)


In [74]:
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_in, d_out_kq, d_out_v):
        super().__init__()
        self.d_out_kq = d_out_kq

        self.W_query = tf.Variable(
            tf.random.uniform((d_in, d_out_kq)), trainable=True
        )
        self.W_key = tf.Variable(
            tf.random.uniform((d_in, d_out_kq)), trainable=True
        )
        self.W_value = tf.Variable(
            tf.random.uniform((d_in, d_out_v)), trainable=True
        )

    def call(self, x):
        keys = tf.matmul(x, self.W_key)      # [T, d_out_kq]
        queries = tf.matmul(x, self.W_query) # [T, d_out_kq]
        values = tf.matmul(x, self.W_value)  # [T, d_out_v]

        # Attention scores: QKᵀ
        attn_scores = tf.matmul(queries, keys, transpose_b=True)  # [T, T]

        # Softmax (scaled by sqrt(d_k))
        attn_weights = tf.nn.softmax(
            attn_scores / tf.math.sqrt(tf.cast(self.d_out_kq, tf.float32)), axis=-1
        )  # [T, T]

        # Weighted sum
        context_vec = tf.matmul(attn_weights, values)  # [T, d_out_v]
        return context_vec

In [75]:
tf.random.set_seed(123)

d_in, d_out_kq, d_out_v = 2, 2, 4

sa = SelfAttention(d_in, d_out_kq, d_out_v)

out = sa(embedded_sentence)

print(out.shape)  # (T, d_out_v)
print(out.numpy())

(4, 4)
[[-0.00133735 -0.00380048 -0.00095654 -0.0047646 ]
 [-0.00141351 -0.00383623 -0.00101422 -0.00479163]
 [-0.00136281 -0.00381231 -0.00097582 -0.00477347]
 [-0.00139955 -0.00382991 -0.00100365 -0.00478699]]


In [76]:
class MultiHeadAttentionWrapper(tf.keras.layers.Layer):
    def __init__(self, d_in, d_out_kq, d_out_v, num_heads):
        super().__init__()
        self.heads = [
            SelfAttention(d_in, d_out_kq, d_out_v)
            for _ in range(num_heads)
        ]

    def call(self, x):
        # รันทุก head แล้ว concat ตามแกนสุดท้าย
        head_outputs = [head(x) for head in self.heads]   # list of [T, d_out_v]
        return tf.concat(head_outputs, axis=-1)           # [T, num_heads * d_out_v]

tf.random.set_seed(123)

d_in, d_out_kq, d_out_v = 2, 2, 1

sa = SelfAttention(d_in, d_out_kq, d_out_v)

# ถ้า embedded_sentence.shape = [T, d_in] เช่น [6, 3]
out = sa(embedded_sentence)

print(out.shape)   # (T, d_out_v) -> (6, 1)
print(out.numpy())

(4, 1)
[[-0.00233419]
 [-0.00238527]
 [-0.00235121]
 [-0.00237601]]


In [77]:
tf.random.set_seed(123)

mha = MultiHeadAttentionWrapper(
    d_in, d_out_kq, d_out_v, num_heads=3
)

# run MHA
context_vecs = mha(embedded_sentence)   # [T, num_heads * d_out_v]

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tf.Tensor(
[[-0.00233419 -0.00181862 -0.00341623]
 [-0.00238527 -0.0018443  -0.00346582]
 [-0.00235121 -0.00182873 -0.00344526]
 [-0.00237601 -0.00183658 -0.00343208]], shape=(4, 3), dtype=float32)
context_vecs.shape: (4, 3)
